In [2]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split



In [3]:
data = pd.read_csv("dataset.csv")
description = pd.read_csv("symptom_Description.csv")
precaution = pd.read_csv("symptom_precaution.csv")
severity = pd.read_csv("Symptom-severity.csv")
print(data.shape)
data.head()

(4920, 18)


,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17
0,Fungal infection,itching,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Fungal infection,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Fungal infection,itching,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Fungal infection,itching,skin_rash,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Fungal infection,itching,skin_rash,nodal_skin_eruptions,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
data = data.fillna("")

for col in data.columns:
    data[col] = data[col].str.strip()
    if col != "Disease":                      
        data[col] = data[col].str.replace(" ", "_")

severity["Symptom"] = severity["Symptom"].str.strip().str.replace(" ", "_")


severity_dict = {}
for i in range(len(severity)):
    severity_dict[severity["Symptom"][i]] = severity["weight"][i]


all_symptoms = sorted(severity_dict.keys())
print("Total symptoms:", len(all_symptoms))

Total symptoms: 132


In [4]:
rows = []
diseases = []

for i in range(len(data)):
    row = {}
    for s in all_symptoms:
        row[s] = 0
    for col in data.columns[1:]:
        s = data[col][i]
        if s in row:
            row[s] = 1
    rows.append(row)
    diseases.append(data["Disease"][i])

X = pd.DataFrame(rows)
y = diseases

X.head()

,abdominal_pain,abnormal_menstruation,acidity,acute_liver_failure,altered_sensorium,anxiety,back_pain,belly_pain,blackheads,bladder_discomfort,...,vomiting,watering_from_eyes,weakness_in_limbs,weakness_of_one_body_side,weight_gain,weight_loss,yellow_crust_ooze,yellow_urine,yellowing_of_eyes,yellowish_skin
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

tree_model = DecisionTreeClassifier()
tree_model.fit(X_train, y_train)

svc_model = SVC()
svc_model.fit(X_train, y_train)

tree_acc = round(tree_model.score(X_test, y_test) * 100, 2)
svc_acc = round(svc_model.score(X_test, y_test) * 100, 2)

print("Decision Tree accuracy:", tree_acc, "%")
print("SVC accuracy          :", svc_acc, "%")

Decision Tree accuracy: 100.0 %
SVC accuracy          : 100.0 %


In [10]:



def get_description(disease):
    d = description[description["Disease"].str.strip() == disease]
    if len(d) == 0:
        return "No description available."
    return d["Description"].values[0]


def get_precautions(disease):
    p = precaution[precaution["Disease"].str.strip() == disease]
    if len(p) == 0:
        return []
    result = []
    for c in ["Precaution_1", "Precaution_2", "Precaution_3", "Precaution_4"]:
        value = p[c].values[0]
        if isinstance(value, str) and value != "":
            result.append(value)
    return result


def find_symptoms(text):
    # Turns 'itching, skin rash' into ['itching', 'skin_rash']
    found = []
    wrong = []
    for word in text.split(","):
        word = word.strip().lower().replace(" ", "_")
        if word == "":
            continue
        if word in all_symptoms:
            found.append(word)
        else:
            wrong.append(word)
    return found, wrong


def predict_disease(symptom_list):
    row = {}
    for s in all_symptoms:
        row[s] = 0
    for s in symptom_list:
        row[s] = 1
    one_row = pd.DataFrame([row])
    return str(tree_model.predict(one_row)[0]), str(svc_model.predict(one_row)[0])

In [7]:
import os

def load_users():
    users = {}
    if os.path.exists("users.csv"):
        f = open("users.csv", "r")
        for line in f:
            line = line.strip()
            if "," in line:
                name, pwd = line.split(",", 1)
                users[name] = pwd
        f.close()
    return users


def register():
    users = load_users()
    name = input("Choose a username: ").strip()
    pwd = input("Choose a password: ").strip()

    if name == "" or pwd == "":
        print("Please fill both username and password.")
    elif "," in name or "," in pwd:
        print("Please do not use commas.")
    elif name in users:
        print("Username already exists.")
    else:
        f = open("users.csv", "a")
        f.write(name + "," + pwd + "\n")
        f.close()
        print("Registered successfully!")


def login():
    users = load_users()
    name = input("Username: ").strip()
    pwd = input("Password: ").strip()

    if name in users and users[name] == pwd:
        print("Login successful!")
        return name
    print("Wrong username or password.")
    return ""


In [12]:
def get_reply(symptoms):
    tree_result, svc_result = predict_disease(symptoms)

    reply = "Decision Tree says: " + tree_result + "\n"
    reply += "SVC says: " + svc_result + "\n"
    if tree_result != svc_result:
        reply += "(The two models differ, please see a doctor.)\n"

    reply += "\nAbout " + tree_result + ":\n" + get_description(tree_result) + "\n"

    precautions = get_precautions(tree_result)
    if len(precautions) > 0:
        reply += "\nPrecautions:\n"
        number = 1
        for p in precautions:
            reply += str(number) + ". " + p + "\n"
            number += 1

    # severity score
    total = 0
    for s in symptoms:
        total += severity_dict[s]
    if total > 13:
        reply += "\nYour symptoms look serious. Please consult a doctor."
    else:
        reply += "\nTake precautions. It may not be serious."

    return reply, tree_result


def chatbot():
    print("Bot: Hello! Type your symptoms separated by commas.")
    print("     Example: itching, skin rash, vomiting")
    print("     Type 'list' to see all symptoms, 'exit' to go back.\n")

    while True:
        text = input("You: ").strip()

        if text.lower() == "exit":
            break

        if text.lower() == "list":
            names = [s.replace("_", " ") for s in all_symptoms]
            print("Bot:", ", ".join(names), "\n")
            continue

        symptoms, wrong = find_symptoms(text)

        if len(wrong) > 0:
            print("Bot: I did not understand:", ", ".join(wrong), "(type 'list' to see valid symptoms)")
        if len(symptoms) == 0:
            continue

        reply, disease = get_reply(symptoms)
        print("\nBot:", reply, "\n")
        

In [ ]:
current_user = ""

while True:
    print("\n===== MEDICAL CHATBOT =====")
    print("1. Register")
    print("2. Login")
    print("3. Quit")
    choice = input("Enter choice: ").strip()

    if choice == "1":
        register()

    elif choice == "2":
        current_user = login()
        if current_user != "":
         
            while True:
                print("\n--- HOME ---  Welcome,", current_user)
                print("Decision Tree accuracy:", tree_acc, "%")
                print("SVC accuracy:", svc_acc, "%")
                print("1. Open Chatbot")
                print("2. Logout")
                home_choice = input("Enter choice: ").strip()
                if home_choice == "1":
                    chatbot()
                elif home_choice == "2":
                    current_user = ""
                    break

    elif choice == "3":
        print("Goodbye!")
        break


===== MEDICAL CHATBOT =====
1. Register
2. Login
3. Quit


Enter choice:  2
Username:  sajin
Password:  sajin


Login successful!

--- HOME ---  Welcome, sajin
Decision Tree accuracy: 100.0 %
SVC accuracy: 100.0 %
1. Open Chatbot
2. Logout


Enter choice:  1


Bot: Hello! Type your symptoms separated by commas.
     Example: itching, skin rash, vomiting
     Type 'list' to see all symptoms, 'exit' to go back.



...

Bot: abdominal pain, abnormal menstruation, acidity, acute liver failure, altered sensorium, anxiety, back pain, belly pain, blackheads, bladder discomfort, blister, blood in sputum, bloody stool, blurred and distorted vision, breathlessness, brittle nails, bruising, burning micturition, chest pain, chills, cold hands and feets, coma, congestion, constipation, continuous feel of urine, continuous sneezing, cough, cramps, dark urine, dehydration, depression, diarrhoea, dischromic patches, distention of abdomen, dizziness, drying and tingling lips, enlarged thyroid, excessive hunger, extra marital contacts, family history, fast heart rate, fatigue, fluid overload, foul smell ofurine, headache, high fever, hip joint pain, history of alcohol consumption, increased appetite, indigestion, inflammatory nails, internal itching, irregular sugar level, irritability, irritation in anus, itching, joint pain, knee pain, lack of concentration, lethargy, loss of appetite, loss of balance, loss of sme

You:  acute liver failure



Bot: Decision Tree says: Cervical spondylosis
SVC says: Fungal infection
(The two models differ, please see a doctor.)

About Cervical spondylosis:
Cervical spondylosis is a general term for age-related wear and tear affecting the spinal disks in your neck. As the disks dehydrate and shrink, signs of osteoarthritis develop, including bony projections along the edges of bones (bone spurs).

Precautions:
1. use heating pad or cold pack
2. exercise
3. take otc pain reliver
4. consult doctor

Take precautions. It may not be serious. 



You:  exit



--- HOME ---  Welcome, sajin
Decision Tree accuracy: 100.0 %
SVC accuracy: 100.0 %
1. Open Chatbot
2. Logout
